# Flights Project: Cleaning the T-100 Destination Data

**Question:** Do U.S. city markets hold different shares of domestic arriving
passengers during winter break than during summer break?

The raw files are loaded with `dtype=str` on purpose. TranStats writes passenger
counts with thousands separators, and letting pandas guess would hide the type
errors discussed in #2. We convert deliberately and check each conversion.

| Failure                 | What it looks like                                                      |
| ----------------------- | ----------------------------------------------------------------------- |
| Duplicates              | Same carrier-market-month twice; passengers overstated                  |
| Type errors             | `6,283` as object, IDs as floats, month as string                       |
| Inconsistent categories | `Denver, CO` / `DENVER, CO` as two destinations                         |
| Impossible values       | Negative passengers, zero distance                                      |
| Missing-not-at-random   | Blank passenger count, but only for one carrier                         |
| **Join fanout**         | Merge silently multiplies rows; nothing errors, every share is now wrong |

### Tell me about my Dataset

In [ ]:
import glob

import numpy as np
import pandas as pd

In [ ]:
# load my datasets - every monthly file, stacked
files = sorted(glob.glob('data/t100_market_*.csv'))
print(files)

In [ ]:
flights = pd.concat([pd.read_csv(f, dtype=str) for f in files], ignore_index=True)
flights = flights.loc[:, ~flights.columns.str.startswith('Unnamed')]
flights.columns = flights.columns.str.upper().str.strip()

In [ ]:
# get data info
flights.info()

In [ ]:
# preview first 10 rows of data
flights.head(10)

In [ ]:
# provide basic data summary statistics for all columns
flights.describe(include='all')

In [ ]:
# summarize frequency of counts of data in categories
for c in ['YEAR', 'MONTH', 'CLASS', 'DATA_SOURCE', 'UNIQUE_CARRIER']:
    print(flights[c].value_counts(dropna=False).head(5))
    print()

In [ ]:
# one row is one carrier's traffic on one market, in one month, for one service class
grain = ['YEAR', 'MONTH', 'UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'CLASS']

### 1: Review for Duplicates

In [ ]:
# how many rows are in the data set?
print(len(flights))

In [ ]:
# how many unique carrier-market-months are in the data?
print(len(flights.drop_duplicates(subset=grain)))

In [ ]:
# how many unique rows in the data?
print(len(flights.drop_duplicates()))

#### Impacts of ignoring duplicate records
Run the next three cells to see the difference between using all rows, all
non-duplicates, or guaranteeing one row per carrier-market-month.

In [ ]:
# sum passengers (all rows)
pax = pd.to_numeric(flights['PASSENGERS'].str.replace(',', '', regex=False),
                    errors='coerce')
print(f"{pax.sum():,.0f}")

In [ ]:
# sum passengers, dropping fully identical rows
pax = pd.to_numeric(flights.drop_duplicates()['PASSENGERS']
                    .str.replace(',', '', regex=False), errors='coerce')
print(f"{pax.sum():,.0f}")

In [ ]:
# sum passengers (one row per carrier-market-month)
pax = pd.to_numeric(flights.drop_duplicates(subset=grain)['PASSENGERS']
                    .str.replace(',', '', regex=False), errors='coerce')
print(f"{pax.sum():,.0f}")

In [ ]:
# which rows are the duplicates? let's look at them
dupe_mask = flights.duplicated(subset=grain, keep=False)
duplicate_rows = flights.loc[dupe_mask].sort_values(grain)
duplicate_rows[grain + ['PASSENGERS']].head(10)

In [ ]:
# keep one row per carrier-market-month
flights = flights.drop_duplicates(subset=grain, keep='first').copy()
print(len(flights))

In [ ]:
# check: did the fix work?
assert len(flights) == len(flights.drop_duplicates(subset=grain))
print("no duplicate carrier-market-months remain")

### 2: Data Types and Special Characters

In [ ]:
# preview passenger values / data types
flights['PASSENGERS'].head()

In [ ]:
# preview the market id values / types
flights['DEST_CITY_MARKET_ID'].head()

In [ ]:
# get the conversion of passengers to numeric values provided by AI
flights['pax'] = (flights['PASSENGERS']
                  .str.replace(',', '', regex=False)
                  .str.strip()
                  )
flights['pax'] = pd.to_numeric(flights['pax'], errors='coerce')

In [ ]:
# convert the rest of what we need
flights['mkt_id'] = pd.to_numeric(flights['DEST_CITY_MARKET_ID'], errors='coerce')
flights['year'] = pd.to_numeric(flights['YEAR'], errors='coerce')
flights['month'] = pd.to_numeric(flights['MONTH'], errors='coerce')
flights['dist'] = pd.to_numeric(flights['DISTANCE'], errors='coerce')
flights[['pax', 'mkt_id', 'year', 'month', 'dist']].dtypes

In [ ]:
# did anything fail to convert that was not already blank?
was_blank = flights['PASSENGERS'].isna() | (flights['PASSENGERS'].str.strip() == '')
broke = flights['pax'].isna() & ~was_blank
print(broke.sum())

In [ ]:
# check: coercion did not invent new missing values
assert broke.sum() == 0
assert pd.api.types.is_numeric_dtype(flights['pax'])
print("passengers converted cleanly")

### 3: Categories that look identical

In [ ]:
# let's look at the information about destination city name
print(flights['DEST_CITY_NAME'].value_counts().head(10))

In [ ]:
# how many spellings does each market id have?
spellings = (flights.groupby('mkt_id')['DEST_CITY_NAME']
             .nunique()
             .sort_values(ascending=False))
print(spellings.head(5))

In [ ]:
# return all the unique values for the worst offender
worst = spellings.index[0]
print([repr(s) for s in flights.loc[flights['mkt_id'] == worst,
                                    'DEST_CITY_NAME'].unique()])

The name column is free text and is not safe to group by, because one metro can be
spelled more than one way and would split into several destinations.

`DEST_CITY_MARKET_ID` is a number assigned by DOT. That is what we group by. The
name is for display only, so we take the most common spelling for each id.

In [ ]:
# cleaning code provided by Claude
canon = (flights.groupby(['mkt_id', 'DEST_CITY_NAME'])
         .size()
         .rename('n')
         .reset_index()
         .sort_values(['mkt_id', 'n'], ascending=[True, False])
         .drop_duplicates(subset='mkt_id', keep='first')
         [['mkt_id', 'DEST_CITY_NAME']]
         .rename(columns={'DEST_CITY_NAME': 'dest_city'}))
canon.head()

In [ ]:
# check: exactly one name per market id
assert canon['mkt_id'].is_unique
print(f"{len(canon)} market ids, one canonical name each")
# follow-up question: what if DOT renames a city market next year?

### 4. Impossible Values

In [ ]:
# passengers cannot be negative and distance cannot be zero
print("negative passengers:  ", (flights['pax'] < 0).sum())
print("zero/neg distance:    ", (flights['dist'] <= 0).sum())
print("month outside 1-12:   ", (~flights['month'].between(1, 12)).sum())

In [ ]:
# look at the impossible rows before dropping them
flights.loc[flights['pax'] < 0, grain + ['pax']].head()

In [ ]:
# drop them - these are not outliers to reason about, they are bad records
before = len(flights)
keep_pax = flights['pax'].isna() | (flights['pax'] >= 0)
keep_dist = flights['dist'].isna() | (flights['dist'] > 0)
flights = flights.loc[keep_pax & keep_dist].copy()
print(f"dropped {before - len(flights)} impossible rows")

In [ ]:
# check: nothing impossible survived
assert (flights['pax'].dropna() >= 0).all()
assert (flights['dist'].dropna() > 0).all()
print("no impossible values remain")

### 5. Missing-not-at-random

In [ ]:
# how many passenger values are blank?
print(flights['pax'].isna().sum())

In [ ]:
# what is the average if we just ignore them?
print(f"{flights['pax'].mean():,.1f}")

In [ ]:
# are the blanks spread evenly across carriers, or concentrated?
print(flights.groupby('UNIQUE_CARRIER')['pax']
      .apply(lambda x: x.isna().mean())
      .sort_values(ascending=False)
      .head(8))

In [ ]:
# and across months?
print(flights.groupby(['year', 'month'])['pax'].apply(lambda x: x.isna().mean()))

If one carrier holds nearly all the blanks, this is not missing at random. Dropping
them removes a slice of the data rather than a random sample, which biases every
share we compute. We drop them anyway, because a blank passenger count cannot be
summed or guessed - but now we know what the bias is and can say so.

In [ ]:
before = len(flights)
flights = flights.dropna(subset=['pax', 'mkt_id']).copy()
print(f"dropped {before - len(flights)} rows with blank passengers or market id")

In [ ]:
# check
assert flights['pax'].notna().all()
print("no blank passenger counts remain")

### 6. Cardinality and Table Joins

In [ ]:
# load the city market lookup table
lookup = pd.read_csv('data/L_CITY_MARKET_ID.csv')
lookup.columns = lookup.columns.str.upper().str.strip()
lookup = lookup.rename(columns={'CODE': 'mkt_id', 'DESCRIPTION': 'metro_name'})
lookup['mkt_id'] = pd.to_numeric(lookup['mkt_id'], errors='coerce')
lookup.head()

In [ ]:
# how many unique codes are in the lookup, and how many rows?
print(lookup['mkt_id'].nunique(), len(lookup))

In [ ]:
# what happens if we merge it as-is?
naive = flights.merge(lookup, on='mkt_id', how='left')
print(len(flights), len(naive))

We went from one row count to a larger one on a left join, which should never
happen. Nothing errored. Every passenger total after this point would be inflated,
and it would look like real traffic.

Two defences: dedupe the right-hand table first, and pass `validate='many_to_one'`
so pandas raises instead of quietly fanning out.

In [ ]:
# dedupe the lookup, then merge safely
lookup = lookup.drop_duplicates(subset='mkt_id', keep='first')
before = len(flights)
flights = flights.merge(lookup, on='mkt_id', how='left', validate='many_to_one')
print(before, len(flights))

In [ ]:
# check: a left join must never change the row count
assert len(flights) == before
print("row count unchanged - no fanout")
print("unmatched market ids:", flights['metro_name'].isna().sum())

### Build one clean dataset

Two filters, each with a reason:

- `CLASS == 'F'` keeps scheduled passenger service and drops charter and all-cargo
  reporting, which are different businesses
- `pax > 0` drops cargo-only records that report zero passengers

The output grain is coarser than the input: one row per destination city market per
month. A new grain needs a new uniqueness check.

In [ ]:
clean = (flights
         .merge(canon, on='mkt_id', how='left', validate='many_to_one')
         .query("CLASS == 'F' and pax > 0")
         .groupby(['year', 'month', 'mkt_id', 'dest_city'], as_index=False)['pax']
         .sum()
         .rename(columns={'pax': 'passengers'})
         .assign(season=lambda d: np.where(d['month'].isin([12, 1]),
                                           'winter', 'summer'))
         .sort_values(['year', 'month', 'passengers'],
                      ascending=[True, True, False])
         .reset_index(drop=True))
clean.head(10)

In [ ]:
# check: is the new grain unique?
print(len(clean), len(clean.drop_duplicates(subset=['year', 'month', 'mkt_id'])))
assert len(clean) == len(clean.drop_duplicates(subset=['year', 'month', 'mkt_id']))

In [ ]:
# check: no blanks, no zero or negative passengers, every month got a season
assert clean.notna().all().all()
assert (clean['passengers'] > 0).all()
assert clean['season'].isin(['winter', 'summer']).all()
print(f"{len(clean)} clean rows, {clean['mkt_id'].nunique()} destination markets")

In [ ]:
# check: did the aggregation conserve the passengers it was given?
source_total = flights.query("CLASS == 'F' and pax > 0")['pax'].sum()
print(f"{source_total:,.0f}  vs  {clean['passengers'].sum():,.0f}")
assert abs(clean['passengers'].sum() - source_total) < 1

In [ ]:
# save the one clean dataset
import os
os.makedirs('data/processed', exist_ok=True)
clean.to_csv('data/processed/destination_month_passengers.csv', index=False)
print("saved", len(clean), "rows")

In [ ]:
# check: the file reads back the same shape it was written
check_file = pd.read_csv('data/processed/destination_month_passengers.csv')
assert check_file.shape == clean.shape
print(check_file.shape)

### Does the question actually work on this data?

Not the analysis yet - just confirming the clean dataset can answer what it was
built for. We use shares rather than raw totals because winter break is two months
and summer is three, so raw totals are not comparable.

In [ ]:
season = (clean.groupby(['season', 'mkt_id', 'dest_city'], as_index=False)
          ['passengers'].sum()
          .assign(share_pct=lambda d: d['passengers']
                  / d.groupby('season')['passengers'].transform('sum') * 100))

In [ ]:
# check: shares have to sum to 100 inside each season
print(season.groupby('season')['share_pct'].sum())
assert np.allclose(season.groupby('season')['share_pct'].sum(), 100)

In [ ]:
index = (season.pivot(index=['mkt_id', 'dest_city'], columns='season',
                      values='share_pct')
         .dropna()
         .assign(seasonality_index=lambda d: d['winter'] / d['summer'])
         .reset_index())

In [ ]:
# most winter-skewed destinations
index.nlargest(10, 'seasonality_index', keep='all')

In [ ]:
# most summer-skewed destinations
index.nsmallest(10, 'seasonality_index', keep='all')

## What I found wrong and what I did about it

*Fill the counts in from the output above after a Restart & Run All.*

| Failure | What I found | What I did |
|---|---|---|
| Duplicates | `___` rows beyond one per carrier-market-month; passenger total overstated by `___` | Dropped repeats on the grain key, keeping the first |
| Type errors | Passengers loaded as text because of thousands separators, so any sum failed silently | Stripped commas, coerced to numeric, checked that no new blanks appeared |
| Inconsistent categories | `___` market ids carried more than one spelling of the city name | Grouped by the DOT market id, never the name; took the most common spelling for display |
| Impossible values | `___` negative passenger counts and `___` zero distances | Dropped the rows instead of adjusting them |
| Missing-not-at-random | `___` blank passenger values, concentrated in carrier `___` | Recorded where they clustered, then dropped them, and noted the bias left behind |
| Join fanout | The lookup table had `___` unique codes across `___` rows, so a plain merge grew the row count | Deduped the lookup first and used `validate='many_to_one'` so a fanout raises |

**The one that would have been invisible.** Every other failure shows up as a strange
number somewhere. Join fanout does not - the merge succeeds, the row count grows, and
the passenger totals rise in a way that looks like real traffic. Without printing the
row count on both sides of the merge there would have been no reason to look.

**What I am still uncertain about.** The blank passenger values were not spread
evenly across carriers, so dropping them slightly under-counts whichever carriers
held them. Because this analysis compares shares between seasons rather than absolute
totals, a carrier missing from both seasons largely cancels out. It would matter if
the blanks were concentrated in one season, which is why the blank rate is printed by
month as well as by carrier.